In [1]:
import pandas as pd
import pykrx

print("환경 설정 완료!")

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
환경 설정 완료!


In [3]:
import OpenDartReader
import matplotlib
import seaborn
import scipy

print("전체 라이브러리 정상 로드 성공!")

전체 라이브러리 정상 로드 성공!


In [8]:
import json
import pandas as pd
import FinanceDataReader as fdr

# 1. config.json 파라미터 불러오기
with open('config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

target_market = config['strategy_params']['market'] # 예: 'KOSPI'
top_n = config['strategy_params']['top_n_mcap']

print(f"{target_market} 시장 데이터를 불러오는 중...")

# 2. FinanceDataReader를 통한 KRX 전종목 리스팅 조회
df_krx = fdr.StockListing('KRX')

# 3. 지정한 시장(KOSPI 등) 필터링 및 시가총액(MarCap) 기준 정렬
top_mcap_df = df_krx[df_krx['Market'] == target_market].sort_values(by='Marcap', ascending=False).head(top_n)

# 4. 보기 좋게 컬럼명 정리 및 출력
top_mcap_df = top_mcap_df[['Code', 'Name', 'Close', 'Marcap', 'Stocks']].rename(
    columns={
        'Code': '종목코드',
        'Name': '종목명',
        'Close': '종가',
        'MarCap': '시가총액',
        'Stocks': '상장주식수'
    }
)

print("\n정상적으로 데이터를 불러왔습니다!")
display(top_mcap_df)

KOSPI 시장 데이터를 불러오는 중...

정상적으로 데이터를 불러왔습니다!


,종목코드,종목명,종가,Marcap,상장주식수
0,005930,삼성전자,249500,1458646512696000,5846278608
1,000660,SK하이닉스,1759000,1253643460035000,712702365
2,402340,SK스퀘어,1109000,146341850074000,131958386
3,005935,삼성전자우,177100,142099940051300,802371203
4,009150,삼성전기,1326000,99043840896000,74693696
5,005380,현대차,401000,82107864166000,204757766
6,373220,LG에너지솔루션,329500,77103000000000,234000000
7,207940,삼성바이오로직스,1518000,70269663618000,46290951
8,032830,삼성생명,318000,63600000000000,200000000
9,105560,KB금융,171500,60828946381000,354687734


In [9]:
import os

# 생성할 프로젝트 디렉토리 및 파일 구조 정의
project_root = "stock_screener"
structure = [
    "config/params.yaml",
    "data/loader.py",
    "stages/__init__.py",
    "stages/stage1_neglected_sector.py",
    "stages/stage2_sector_leaders.py",
    "stages/stage3_fundamental_improve.py",
    "stages/stage4_valuation.py",
    "stages/stage5_financial_health.py",
    "core/pipeline.py",
    "core/schema.py",
    "backtest/forward_return.py",
    "main.py"
]

# 폴더 및 빈 파일 자동 생성
for path in structure:
    full_path = os.path.join(project_root, path)
    
    # 상위 폴더 생성
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    
    # 빈 파일 생성 (이미 존재하면 건너뜀)
    if not os.path.exists(full_path):
        with open(full_path, 'w', encoding='utf-8') as f:
            pass 

print(f"'{project_root}' 프로젝트 뼈대 생성 완료!")

'stock_screener' 프로젝트 뼈대 생성 완료!


In [4]:
import pandas as pd
from datetime import date
from data.loader import QuantDataLoader

def test_data_loader():
    print("==================================================")
    print("🚀 QuantDataLoader 테스트를 시작합니다...")
    print("==================================================\n")
    
    # 1. 로더 인스턴스 생성
    try:
        print("[테스트 1] 로더 인스턴스화 및 환경변수 확인")
        loader = QuantDataLoader(use_cache=True)
        print("✅ 성공: DART API 키 및 로더 초기화 완료!\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")
        return

    # 2. 유니버스 로드 테스트 (Point-in-Time)
    # 현재 2026년 7월 26일이므로, 최근 거래일인 2026년 7월 24일(금요일)로 테스트합니다.
    test_date = date(2023, 7, 24)
    print(f"[테스트 2] KOSPI 유니버스 데이터 로드 ({test_date})")
    try:
        universe_df = loader.get_kospi_universe(test_date)
        print(f"✅ 성공: 총 {len(universe_df)}개 종목 로드 완료!")
        print("-" * 50)
        print(universe_df.head().to_string(index=False))
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    # 3. DART 재무제표 파싱 테스트
    # 대표 종목인 삼성전자(005930)의 2025년도 사업보고서(11011)를 요청합니다.
    ticker_to_test = '005930'
    target_year = 2023
    print(f"[테스트 3] {ticker_to_test} {target_year}년 사업보고서(11011) 파싱")
    try:
        financials = loader.parse_standardized_financials(ticker_to_test, target_year, '11011')
        print(f"✅ 성공: 재무 데이터 표준화 완료!")
        print("-" * 50)
        for key, value in financials.items():
            # float('nan') 처리를 위해 pd.isna 확인 후 출력 포맷팅
            if pd.isna(value):
                print(f"{key:>20} : NaN")
            else:
                print(f"{key:>20} : {value:,.0f}")
        print("-" * 50 + "\n")
    except Exception as e:
        print(f"❌ 실패: {e}\n")

    print("==================================================")
    print("🎯 모든 테스트가 종료되었습니다.")
    print("==================================================")

if __name__ == "__main__":
    test_data_loader()

🚀 QuantDataLoader 테스트를 시작합니다...

[테스트 1] 로더 인스턴스화 및 환경변수 확인
✅ 성공: DART API 키 및 로더 초기화 완료!

[테스트 2] KOSPI 유니버스 데이터 로드 (2023-07-24)
Error occurred in get_market_cap_by_ticker: Expecting value: line 1 column 1 (char 0)
❌ 실패: "None of [Index(['종가', '시가총액', '거래량', '거래대금'], dtype='object')] are in the [columns]"

[테스트 3] 005930 2023년 사업보고서(11011) 파싱
❌ 실패: 'fs_div'

🎯 모든 테스트가 종료되었습니다.
